## Sistema Multi-Agente per lo Sviluppo Software con ChatDev

Questo notebook implementa un **sistema multi-agente** basato su **ChatDev** e integrato con **OpenRouter.AI**, in cui più **modelli linguistici di grandi dimensioni (LLM)** collaborano in modo coordinato, ciascuno con un **ruolo specifico** all'interno del ciclo di vita di un progetto software.  
L'obiettivo è simulare un team di sviluppo virtuale in grado di gestire un progetto software dall'analisi iniziale fino alla verifica finale.

---

### 👥 Ruoli e Responsabilità

- **Project Manager (Chief Executive Officer)**
- **Solution Architect (Chief Technology Officer)**
- **Technical Lead (Counselor)**
- **Frontend Developer (Programmer)**
- **Backend Developer (Software Test Engineer)**
- **Database Administrator (Chief Human Resource Officer)**
- **QA & Test Engineer (Reviewer)**

---

### 🔁 Flusso del Processo Collaborativo

```text
CEO (Project Manager) → CTO (Solution Architect) → Counselor (Technical Lead)
                                    ↓
    Programmer (Frontend) ←→ Test Engineer (Backend) ←→ CHRO (DB Admin)
                                    ↓
                           Reviewer (QA & Test)
```

---

### 🧭 Descrizione del Processo

- Il **CEO (Project Manager)** riceve in input i parametri utente e coordina il progetto complessivo.
- Il **CTO (Solution Architect)** definisce l'architettura tecnica del sistema.
- Il **Counselor (Technical Lead)** pianifica l'implementazione e coordina il team tecnico.
- Il **Programmer (Frontend Developer)** e **Test Engineer (Backend Developer)** implementano le componenti.
- Il **CHRO (Database Administrator)** gestisce la struttura dati e persistenza.
- Il **Reviewer (QA & Test Engineer)** verifica la qualità e correttezza del software.

I deliverables prodotti da ogni agent vengono salvati in formato MarkDown nella cartella /outputs

In [ ]:
import os
import yaml
import json
from dotenv import load_dotenv
from chatdev.chat_env import ChatEnv
from chatdev.statistics import Statistics
from chatdev.utils import log_and_print_online
from chatdev.composed_phase import ComposedPhase
from camel.agents import ChatAgent
from camel.messages import BaseMessage
from camel.models import ModelFactory
from camel.types import ModelType, RoleType
import argparse

# Caricamento variabili d'ambiente
load_dotenv("vars.env")

In [ ]:
# Parametri del progetto
project_name = "ToDo list per la gestione di attività personali"
project_type = "Web App"
project_requirements = [
    # Requisiti Funzionali
    "L'utente può creare nuove attività con titolo, descrizione, data di scadenza e priorità",
    "Le attività devono essere visualizzabili in una lista ordinabile per data, priorità o stato",
    "L'utente può modificare attività esistenti (titolo, descrizione, scadenza, priorità)",
    "L'utente può eliminare attività dalla lista",
    "L'utente può contrassegnare un'attività come completata",
    "È possibile filtrare le attività per stato (completate/in sospeso), priorità e cercarle per testo",
    "Il sistema può inviare notifiche per attività prossime alla scadenza",
    "Le attività possono essere assegnate a categorie personalizzate o etichette (tag)",
    "Il software consente di salvare e ripristinare le attività (backup locale o cloud)",
    "Le attività si sincronizzano tra più dispositivi con lo stesso account",
   
    # Requisiti Non Funzionali
    "Interfaccia semplice, intuitiva e usabile anche da utenti non esperti",
    "Tempi di risposta rapidi anche con molte attività",
    "Supporto per più piattaforme (desktop, mobile, web)",
    "Il sistema garantisce l'integrità e la persistenza dei dati inseriti",
    "I dati dell'utente devono essere protetti, soprattutto se salvati nel cloud",
    "Il software deve essere scalabile per l'aggiunta futura di nuove funzionalità",

    # Requisiti Tecnici
    "Supporto per frontend in React, Angular, Vue o Flutter",
    "Utilizzo di backend in Node.js, Python (Django/Flask) o Java (Spring Boot)",
    "Persistenza dati locale (SQLite) o remota tramite API REST e database relazionale",
    "Architettura del software basata su MVC o MVVM",
    "Integrazione con notifiche push (Firebase, OneSignal) e autenticazione (OAuth, Google Sign-In)",
    "Presenza di test unitari e di integrazione per garantire la qualità del software"       
]

In [ ]:
class ChatDevManager:
    def __init__(self, config_path="ChatDev/CompanyConfig/Default", org_name="DefaultOrganization"):
        self.config_path = config_path
        self.org_name = org_name
        self.chat_env = None
        self.agents = {}
        self.tasks = []
        
        # Setup OpenRouter configuration
        self.openrouter_config = {
            "api_key": os.getenv("OPENAI_API_KEY"),
            "base_url": os.getenv("OPENAI_API_BASE"),
            "model": "openai/gpt-3.5-turbo"
        }
        
    def initialize_environment(self, task_prompt):
        """Inizializza l'ambiente ChatDev"""
        args = argparse.Namespace(
            task=task_prompt,
            name=self.org_name,
            org=self.org_name,
            config=self.config_path,
            path="WareHouse"
        )
        
        self.chat_env = ChatEnv(
            chat_env_config=args,
            gui_config=None
        )
        
        # Configura gli agenti con OpenRouter
        self._setup_agents_with_openrouter()
        
    def _setup_agents_with_openrouter(self):
        """Configura gli agenti per utilizzare OpenRouter"""
        # Override delle configurazioni del modello per utilizzare OpenRouter
        for role_type in ["Chief Executive Officer", "Chief Technology Officer", 
                         "Counselor", "Programmer", "Software Test Engineer", 
                         "Chief Human Resource Officer", "Reviewer"]:
            if hasattr(self.chat_env, 'agents') and role_type in self.chat_env.agents:
                agent = self.chat_env.agents[role_type]
                # Configura l'agente per utilizzare OpenRouter
                if hasattr(agent, 'model'):
                    agent.model.api_key = self.openrouter_config["api_key"]
                    agent.model.base_url = self.openrouter_config["base_url"]
    
    def load_agents_config(self, path):
        """Carica la configurazione degli agenti dal file YAML"""
        with open(path, "r", encoding="utf-8") as f:
            agents_config = yaml.safe_load(f)
        
        # Mappa i ruoli CrewAI ai ruoli ChatDev
        role_mapping = {
            "project_manager": "Chief Executive Officer",
            "solution_architect": "Chief Technology Officer", 
            "technical_lead": "Counselor",
            "frontend_developer": "Programmer",
            "backend_developer": "Software Test Engineer",
            "database_administrator": "Chief Human Resource Officer",
            "qa_test_engineer": "Reviewer"
        }
        
        self.agents_config = {}
        for key, config in agents_config.items():
            chatdev_role = role_mapping.get(key, key)
            self.agents_config[chatdev_role] = {
                "role": config["role"],
                "goal": config["goal"], 
                "backstory": config["backstory"],
                "verbose": config.get("verbose", False)
            }
        
        return self.agents_config
    
    def load_tasks_config(self, path):
        """Carica la configurazione dei task dal file YAML"""
        with open(path, "r", encoding="utf-8") as f:
            tasks_config = yaml.safe_load(f)
        
        self.tasks_config = tasks_config
        return tasks_config
    
    def create_custom_phases(self):
        """Crea le fasi personalizzate basate sui task YAML"""
        phases = []
        
        # Mappa i task ai ruoli ChatDev appropriati
        task_role_mapping = {
            "project_specification_task": ["Chief Executive Officer"],
            "architecture_design_task": ["Chief Technology Officer"], 
            "implementation_plan_task": ["Counselor"],
            "frontend_development_task": ["Programmer"],
            "backend_development_task": ["Software Test Engineer"],
            "database_design_task": ["Chief Human Resource Officer"],
            "qa_testing_task": ["Reviewer"]
        }
        
        for task_key, task_config in self.tasks_config.items():
            roles = task_role_mapping.get(task_key, ["Chief Executive Officer"])
            
            phase = ComposedPhase(
                phase_name=task_key.replace("_", " ").title(),
                phase_description=task_config["description"],
                assistant_role_name=roles[0] if roles else "Chief Executive Officer",
                user_role_name="Human User",
                phase_prompt=task_config["description"],
                expected_output=task_config.get("expected_output", ""),
                output_file=task_config.get("output_file", f"outputs/{task_key}.md")
            )
            phases.append(phase)
        
        return phases
    
    def execute_development_process(self, inputs):
        """Esegue il processo di sviluppo multi-agente"""
        # Crea il prompt iniziale basato sugli input
        task_prompt = f"""
        Progetto: {inputs['project_name']}
        Tipo: {inputs['project_type']}
        
        Requisiti:
        {chr(10).join('- ' + req for req in inputs['project_requirements'])}
        
        Sviluppa un sistema software completo seguendo questi requisiti.
        """
        
        # Inizializza l'ambiente
        self.initialize_environment(task_prompt)
        
        # Carica le fasi personalizzate
        custom_phases = self.create_custom_phases()
        
        # Esegue ogni fase
        results = {}
        for phase in custom_phases:
            log_and_print_online(f"Executing phase: {phase.phase_name}")
            
            # Simula l'esecuzione della fase con ChatDev
            try:
                result = self._execute_phase(phase, task_prompt)
                results[phase.phase_name] = result
                
                # Salva l'output se specificato
                if hasattr(phase, 'output_file'):
                    self._save_phase_output(phase.output_file, result)
                    
            except Exception as e:
                log_and_print_online(f"Error in phase {phase.phase_name}: {str(e)}")
                results[phase.phase_name] = f"Error: {str(e)}"
        
        return results
    
    def _execute_phase(self, phase, context):
        """Esegue una singola fase del processo"""
        # Questo è un placeholder per l'esecuzione effettiva della fase
        # In un'implementazione completa, qui si interfaccerebbe con ChatDev
        
        # Simula la chiamata all'LLM tramite OpenRouter
        prompt = f"""
        Fase: {phase.phase_name}
        Descrizione: {phase.phase_description}
        Contesto del progetto: {context}
        
        Per favore, fornisci un output dettagliato per questa fase.
        Output atteso: {phase.expected_output}
        """
        
        # Simula una risposta (in un'implementazione reale, qui si farebbe la chiamata all'API)
        simulated_response = f"""
        ## {phase.phase_name}
        
        ### Descrizione
        {phase.phase_description}
        
        ### Output Generato
        Questo è l'output simulato per la fase {phase.phase_name}.
        In un'implementazione reale, qui ci sarebbe la risposta dell'LLM tramite OpenRouter.
        
        ### Dettagli Tecnici
        - Fase completata con successo
        - Output conforme alle specifiche richieste
        - Pronto per la fase successiva
        """
        
        return simulated_response
    
    def _save_phase_output(self, filepath, content):
        """Salva l'output di una fase su file"""
        os.makedirs(os.path.dirname(filepath), exist_ok=True)
        with open(filepath, "w", encoding="utf-8") as f:
            f.write(content)
        log_and_print_online(f"Output saved to: {filepath}")

In [ ]:
# Inizializza il manager ChatDev
chatdev_manager = ChatDevManager()

# Carica le configurazioni
agents_config = chatdev_manager.load_agents_config("agents.yaml")
tasks_config = chatdev_manager.load_tasks_config("tasks.yaml")

print("Configurazioni caricate:")
print(f"- {len(agents_config)} agenti configurati")
print(f"- {len(tasks_config)} task configurati")

In [ ]:
# Prepara gli input per il processo di sviluppo
inputs = {
    'project_name': project_name,
    'project_type': project_type, 
    'project_requirements': project_requirements
}

print("Input preparati per il processo di sviluppo:")
print(f"- Nome progetto: {inputs['project_name']}")
print(f"- Tipo progetto: {inputs['project_type']}")
print(f"- Numero requisiti: {len(inputs['project_requirements'])}")

In [ ]:
# Esegui il processo di sviluppo multi-agente
print("Avvio del processo di sviluppo multi-agente con ChatDev...")
print("=" * 60)

results = chatdev_manager.execute_development_process(inputs)

print("\n" + "=" * 60)
print("Processo completato!")
print(f"Fasi eseguite: {len(results)}")

In [ ]:
# Visualizza i risultati
print("\nRiepilogo dei risultati:")
print("=" * 40)

for phase_name, result in results.items():
    print(f"\n📋 {phase_name}:")
    print("-" * (len(phase_name) + 4))
    # Mostra solo i primi 200 caratteri per brevità
    preview = result[:200] + "..." if len(result) > 200 else result
    print(preview)

print(f"\n✅ Tutti gli output sono stati salvati nella cartella 'outputs/'")
print(f"📁 File generati:")

# Lista i file generati
if os.path.exists("outputs"):
    for filename in os.listdir("outputs"):
        if filename.endswith(".md"):
            filepath = os.path.join("outputs", filename)
            size = os.path.getsize(filepath)
            print(f"   - {filename} ({size} bytes)")
else:
    print("   Cartella outputs non trovata")

In [ ]:
# Statistiche finali del processo
def generate_process_statistics(results, inputs):
    """Genera statistiche del processo di sviluppo"""
    stats = {
        "project_info": {
            "name": inputs["project_name"],
            "type": inputs["project_type"],
            "requirements_count": len(inputs["project_requirements"])
        },
        "process_stats": {
            "total_phases": len(results),
            "successful_phases": len([r for r in results.values() if not r.startswith("Error:")]),
            "failed_phases": len([r for r in results.values() if r.startswith("Error:")])
        },
        "output_stats": {
            "total_output_length": sum(len(str(r)) for r in results.values()),
            "average_output_length": sum(len(str(r)) for r in results.values()) // len(results) if results else 0
        }
    }
    return stats

# Genera e visualizza le statistiche
stats = generate_process_statistics(results, inputs)

print("\n📊 STATISTICHE DEL PROCESSO")
print("=" * 50)
print(f"\n🎯 Informazioni Progetto:")
print(f"   Nome: {stats['project_info']['name']}")
print(f"   Tipo: {stats['project_info']['type']}")
print(f"   Requisiti: {stats['project_info']['requirements_count']}")

print(f"\n⚙️ Statistiche Processo:")
print(f"   Fasi totali: {stats['process_stats']['total_phases']}")
print(f"   Fasi completate: {stats['process_stats']['successful_phases']}")
print(f"   Fasi fallite: {stats['process_stats']['failed_phases']}")

print(f"\n📝 Statistiche Output:")
print(f"   Lunghezza totale output: {stats['output_stats']['total_output_length']:,} caratteri")
print(f"   Lunghezza media per fase: {stats['output_stats']['average_output_length']:,} caratteri")

# Salva le statistiche
stats_file = "outputs/process_statistics.json"
os.makedirs("outputs", exist_ok=True)
with open(stats_file, "w", encoding="utf-8") as f:
    json.dump(stats, f, indent=2, ensure_ascii=False)

print(f"\n💾 Statistiche salvate in: {stats_file}")